# Training

## Student

In [1]:
# !pip install torchinfo
import torch
import torch.nn as nn
from torchinfo import summary
import matplotlib.pyplot as plt
import os
import gc
import numpy as np
import time
import sys
sys.path.insert(1, './Alpha')
import Alpha.Student_LSTM_aggr as TP
import Datasetting.DataOrganizer_old as DS
print(torch.cuda.is_available())

True


In [2]:
date = '2024_Door_RGB/Alpha/'
name = f'{date}Student_3V'
data_path = [
    '../dataset/Door_EXP/TrainVer/A208',
            ]
level = 'env'

crossvalidator = DS.CrossValidator(level, all_range=[None], subset_ratio=0.2)
data_organizer = DS.DataOrganizer(name, data_path, crossvalidator)
data_organizer.load()

Cross validation plan at env level
Loading ../dataset/Door_EXP/TrainVer/A208...

Loaded 0709A02-csi.npy
Loaded 0709A04-pd.npy
Loaded 0709A10-pd.npy
Loaded 0709A11-csi.npy
Loaded 0709A12-csi.npy
Loaded 0709A14-csi.npy
Loaded 0709A20-csi.npy
Loaded 0709A23-csi.npy
Loaded 0709A24-csi.npy
Loaded 0709A40-csi.npy
Loaded 0709A40-pd.npy
Loaded 0709A41-csi.npy
Loaded 0709A50-csi.npy
Loaded 0709A50-pd.npy
Loaded 0709A511-csi.npy
Loaded 0709A531-pd.npy
Loaded 20240709_151439-dpt.npy
Loaded 20240709_152129-dpt.npy
Loaded 20240709_152129-ctr.npy
Loaded 20240709_153519-cimg.npy
Loaded 20240709_153519-rimg.npy
Loaded 20240709_153810-ctr.npy
Loaded 20240709_153959-cimg.npy
Loaded 20240709_153959-dpt.npy
Loaded 20240709_153959-ctr.npy
Loaded 20240709_154427-cimg.npy
Loaded 20240709_154427-dpt.npy
Loaded 20240709_154427-ctr.npy
Loaded 20240709_154950-rimg.npy
Loaded 20240709_155837-cimg.npy
Loaded 20240709_155837-ctr.npy
Loaded 20240709_155837-dpt.npy
Loaded 20240709_161523-dpt.npy
Loaded 20240709_16180

/mnt/datastore/Models/Datasetting/DataOrganizer_old.py:181: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.total_segment_labels = pd.concat([self.total_segment_labels, sub_label], ignore_index=True)


Loaded 0709A22-csi.npy
Loaded 0709A34-pd.npy
Loaded 0709A241-pd.npy
Loaded 0709A4231-pd.npy
Loaded 0709A32-csi.npy
Loaded 0709A422-pd.npy
Loaded 0709A13-csi.npy
Loaded 0709A33-csi.npy
Loaded 0709A422-csi.npy
Loaded 0709A10-csi.npy
Loaded 0709A231-csi.npy
Loaded 0709A34-csi.npy
Loaded 0709A4231-csi.npy
Loaded 0709A12-pd.npy
Loaded 0709A511-pd.npy
Loaded 20240709_151047-cimg.npy
Loaded 20240709_151047-ctr.npy
Loaded 20240709_151047-rimg.npy
Loaded 20240709_153519-ctr.npy
Loaded 20240709_153810-cimg.npy
Loaded 20240709_153959-rimg.npy
Loaded 20240709_155258-ctr.npy
Loaded 20240709_160253-dpt.npy
Loaded 20240709_160253-ctr.npy
Loaded 20240709_160646-dpt.npy
Loaded 20240709_160646-ctr.npy
Loaded 20240709_160857-cimg.npy
Loaded 20240709_160857-rimg.npy
Loaded 20240709_161523-cimg.npy
Loaded 20240709_161802-cimg.npy
Loaded 20240709_161802-dpt.npy
Loaded 20240709_161802-ctr.npy
Loaded 20240709_162037-ctr.npy
Loaded 20240709_162151-ctr.npy
Loaded 20240709_163045-cimg.npy
Loaded 20240709_163401-

In [3]:
data_organizer.load_plan('../dataset/Door_EXP/subject_r0.2_A208.pkl')
data_organizer.gen_plan()

Data Organizer: Loaded plan!


In [4]:
train_loader, valid_loader, test_loader, current_test = data_organizer.gen_loaders(num_workers=2, batch_size=64)

Data Organizer: Generating loaders for s: current test = higashinaka
 Train/Valid dataset length = 9503
 Test dataset length = 2027
 Exported train loader of len 118, batch size = 64

 Exported valid loader of len 29, batch size = 64

 Exported test loader of len 31, batch size = 64



In [ ]:
preprocess = DS.Preprocess(new_size=(128, 128))
gpu = 0
torch.cuda.set_device(gpu)
S_trainer = TP.StudentTrainer(name='Student',
                              lstm_steps = 75,
                              lr=1e-4, epochs=50, cuda=gpu,
                              preprocess = preprocess,
                              notion=f'{name}_hp_1',
                              dataloaders={
                                  'train': train_loader,
                                  'valid': valid_loader,
                                  'test': test_loader
                              },
                             )

S_trainer.load(f"../saved/2024_Door_EXP/Teachers/Prop/20240911_Sub_Prop_higashinaka", name='Teacher', mode='best')
S_trainer.schedule(early_stop=True, lr_decay=False)

==========2024_Door_RGB/Alpha/Student_3V_hp_1 Student Loading==========
Loading ../saved/2024_Door_EXP/Teachers/Prop/20240911_Sub_Prop_higashinaka
Loaded model imgen from ../saved/2024_Door_EXP/Teachers/Prop/20240911_Sub_Prop_higashinaka/Teacher_imgen_best.pth!
Loaded model cimgde from ../saved/2024_Door_EXP/Teachers/Prop/20240911_Sub_Prop_higashinaka/Teacher_cimgde_best.pth!
Loaded model rimgde from ../saved/2024_Door_EXP/Teachers/Prop/20240911_Sub_Prop_higashinaka/Teacher_rimgde_best.pth!
Loaded model ctrde from ../saved/2024_Door_EXP/Teachers/Prop/20240911_Sub_Prop_higashinaka/Teacher_ctrde_best.pth!
==========2025-05-12 12:30:21 2024_Door_RGB/Alpha/Student_3V_hp_1 Student Training starting==========


  0%|                                                                                        | 1/1001 [00:00<?…

  0%|                                                                                                       |[…

main csien cnn.0.weight: requires_grad=True
main csien cnn.0.bias: requires_grad=True
main csien cnn.3.weight: requires_grad=True
main csien cnn.3.bias: requires_grad=True
main csien cnn.6.weight: requires_grad=True
main csien cnn.6.bias: requires_grad=True
main csien lstm.weight_ih_l0: requires_grad=True
main csien lstm.weight_hh_l0: requires_grad=True
main csien lstm.bias_ih_l0: requires_grad=True
main csien lstm.bias_hh_l0: requires_grad=True
main csien lstm.weight_ih_l1: requires_grad=True
main csien lstm.weight_hh_l1: requires_grad=True
main csien lstm.bias_ih_l1: requires_grad=True
main csien lstm.bias_hh_l1: requires_grad=True
main csien fc_feature.0.weight: requires_grad=True
main csien fc_feature.0.bias: requires_grad=True
main csien fc_pd.0.weight: requires_grad=True
main csien fc_pd.0.bias: requires_grad=True
main csien fc_mu.0.weight: requires_grad=True
main csien fc_mu.0.bias: requires_grad=True
main csien fc_logvar.0.weight: requires_grad=True
main csien fc_logvar.0.bias:

  0%|                                                                                                       |[…

In [ ]:
a = 1